<h1><center>Hands-on RL: land a SpaceX rocket</center></h1>

The goal is to **land a rocket on its ship, consistently**. You will train an agent with
PPO (or A2C) from Stable-Baselines3 and, above all, **design its reward**: the reward is
what shapes the behaviour of the rocket.

Your agent is scored on how often it lands out of 100 episodes and how close to the
center of the ship.

**Setup (once):** in a terminal, `cd` into the repository and run `uv sync`, then select
the `.venv` kernel of this repository in your notebook editor.

Prefer clicking? The same exercises exist in the web app: `uv run python -m rocketlander.app`.

# Setup

In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Video, display
from stable_baselines3 import A2C, PPO
from stable_baselines3.common.callbacks import CallbackList, EvalCallback
from stable_baselines3.common.vec_env import VecNormalize

from rocketlander import make_env
from rocketlander.training import LandingStatsCallback, evaluate, make_vec, record_episode

RUNS_DIR = Path("../runs")
VIDEOS_DIR = Path("../videos")


def linear_schedule(initial_value: float, final_value: float):
    """Learning rate going linearly from initial_value to final_value during training."""
    def func(progress_remaining: float) -> float:  # 1 at the beginning, 0 at the end
        return final_value + progress_remaining * (initial_value - final_value)
    return func


def show(path, width=320):
    display(Video(str(path), embed=True, width=width, html_attributes="controls autoplay loop"))

# The environment

**Actions** (continuous mode, each value in [-1, 1]):

| index | action | effect |
|---|---|---|
| 0 | gimbal | turns the main engine left / right |
| 1 | throttle | increases / decreases the main engine power |
| 2 | side thruster | < -0.5 fires left, > 0.5 fires right |

In discrete mode there are 7 actions: gimbal +/-, throttle +/-, left thruster, right
thruster, do nothing.

**What your reward function receives** (`s` is a dict, values computed at every step):

| key | meaning |
|---|---|
| `x` | horizontal position, -1 (left edge) .. 1 (right edge), 0 = ship center |
| `x_on_ship` | position on the ship: 0 = center, -1 / +1 = ship edges |
| `altitude` | height above the deck, in meters |
| `rotation` | number of turns since the start (+ = counter-clockwise) |
| `y` | height, -1 = ship deck .. about 1 = start height |
| `angle` | tilt in units of pi, 0 = upright, negative = left |
| `left_leg_contact`, `right_leg_contact` | is the leg touching the ship |
| `velocity_x`, `velocity_y` | speed (negative `velocity_y` = falling) |
| `speed`, `angular_velocity`, `distance` | overall speed, rotation speed, distance to ship center |
| `throttle`, `gimbal`, `power`, `side_thruster` | engine state |
| `fuel` | 1 = full tank, 0 = empty |
| `touch_down` | a leg touched the ship at some point in this episode |
| `landed` | both legs on the ship and slow |
| `landed_full` | landed for 1 second: **success**, the episode ends |
| `game_over` | crash (hull touched something, broken leg, left the screen) |
| `broken_leg`, `outside` | why it crashed |
| `step` | physics frame of the episode (60 per second) |
| `memory` | a dict you can use to remember things during the episode (emptied at each new episode) |

**Difficulty levels**

- `0`: legs never break (even crashing at full speed on the ship "lands"). Debugging only.
- `1-9`: legs break above a touchdown speed: 7.6 m/s at level 1 ... 4.4 m/s at level 9;
  the start position gets more random.
- `10`: same limit, and legs also break on a hard landing (leg joints over-compressed).
- `11-20`: same, and the legs get shorter and shorter.

On top of the level, `CONFIG` below overrides any scenario parameter: `ship_width` (zone
size), `start_x`, `start_height`, `start_speed`, `max_touchdown_speed`, `leg_scale`,
`end_on_landing`, `fuel`... (see `DEFAULT_CONFIG` in `gym/envs/box2d/rocket_lander.py`).

![explanation](https://github.com/YannBerthelot/hackathon_RL_XHEC_2020/blob/master/images/explanation.png?raw=1)

The environment is based on https://github.com/EmbersArc/gym_rocketLander

## Look at a random agent

Before training anything, watch what a random policy does.

In [ ]:
class RandomAgent:
    def __init__(self, env):
        self.action_space = env.action_space

    def predict(self, obs, deterministic=True):
        return self.action_space.sample(), None


path, traj = record_episode(RandomAgent(make_env()), VIDEOS_DIR / "random.mp4", level=0)
print("steps:", len(traj), "| crash:", traj[-1]["game_over"], "| landed:", traj[-1]["landed_full"])
show(path)

# Shape your reward

`my_reward(s, base_reward)` is called at **every physics frame** (60 per second). `s` is the dict described above
and `base_reward` is the reward computed by the environment (it works: an agent trained on
it learns to land).

- `return base_reward` keeps the default behaviour.
- Add terms to it to change the behaviour (land closer to the center, save fuel, land
  faster, stay upright...), or replace it completely.

Tips:
- Keep the rewards of one step small (roughly between -1 and 1), and put big bonuses /
  penalties only on terminal events (`landed_full`, `game_over`).
- Watch out for loopholes: if the agent can collect reward forever without landing, it
  will do exactly that.

*!!! Change `CONFIG_NAME` (below) whenever you change something, so your runs are easy to
compare in TensorBoard !!!*

---
*RESTART FROM HERE WHEN CHANGING YOUR REWARD*

---

In [ ]:
def my_reward(s, base_reward):
    reward = base_reward  # start from the default reward, or set reward = 0 to start from scratch

    # Examples, uncomment / adapt:

    # Stay upright
    # reward -= 0.01 * abs(s["angle"])

    # Land in the middle of the ship
    # if s["landed_full"]:
    #     reward += 0.5 * (1 - abs(s["x"]))

    # Save fuel
    # if s["landed_full"]:
    #     reward += 0.5 * s["fuel"]

    # Don't go up
    # reward -= 0.01 * max(0.0, s["velocity_y"])

    # Extra penalty for crashing
    # if s["game_over"]:
    #     reward -= 1

    return reward


REWARD_FN = my_reward  # set to None to use the default reward

## Environment creation

`N_ENVS` environments run in parallel to collect experience faster.

In [ ]:
N_ENVS = 16
CONTINUOUS = True
LEVEL = 1
# Easier start (lower, slower): learns in minutes on a laptop. Remove it for the real thing.
CONFIG = {"start_height": 0.35, "start_speed": 10.0, "start_spin": 0.3}

env = make_vec(level=LEVEL, continuous=CONTINUOUS, reward_fn=REWARD_FN, n_envs=N_ENVS, seed=0, config=CONFIG)
eval_env = make_vec(level=LEVEL, continuous=CONTINUOUS, reward_fn=REWARD_FN, n_envs=1, seed=1000, config=CONFIG)
# Scale the rewards during training only (keeps the value function well conditioned)
env = VecNormalize(env, norm_obs=False, norm_reward=True, gamma=0.999)
eval_env = VecNormalize(eval_env, norm_obs=False, norm_reward=False, training=False)

# Agent definition

Tune the hyperparameters and study their influence on training.

- **PPO** learns more slowly but more steadily, and gives the best results when well tuned.
- **A2C** usually learns faster but is a lot less stable: handy to quickly see what
  behaviour a new reward induces.

Documentation: https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html

---
*RESTART FROM HERE WHEN CHANGING YOUR AGENT*

---

In [ ]:
MODEL_ALGORITHM = "PPO"  # or "A2C"

if MODEL_ALGORITHM == "PPO":
    # defaults from RL Zoo for LunarLanderContinuous, a very similar task
    N_STEPS = 1024    # steps collected per env before each update
    BATCH_SIZE = 64   # minibatch size
    N_EPOCH = 4       # passes over the collected data at each update
    GAMMA = 0.999     # discount factor: how much the future matters
    GAE = 0.98        # Generalized Advantage Estimation lambda
    CLIP = 0.2        # lower = more conservative policy updates
    ENT = 0.01        # entropy bonus: > 0 encourages exploration
    VF = 0.5          # weight of the value function loss
    MAX_NORM = 0.5    # gradient clipping
    LR_START, LR_END = 3e-4, 3e-5

    model = PPO(
        "MlpPolicy", env,
        learning_rate=linear_schedule(LR_START, LR_END),
        n_steps=N_STEPS, batch_size=BATCH_SIZE, n_epochs=N_EPOCH,
        gamma=GAMMA, gae_lambda=GAE, clip_range=CLIP,
        ent_coef=ENT, vf_coef=VF, max_grad_norm=MAX_NORM,
        tensorboard_log=str(RUNS_DIR), seed=0, verbose=0,
    )
    CONFIG_NAME = f"PPO_cont{CONTINUOUS}_level{LEVEL}_steps{N_STEPS}_batch{BATCH_SIZE}_gamma{GAMMA}_ent{ENT}_LR{LR_START}"
else:
    model = A2C("MlpPolicy", env, gamma=0.999, tensorboard_log=str(RUNS_DIR), seed=0, verbose=0)
    CONFIG_NAME = f"A2C_cont{CONTINUOUS}_level{LEVEL}"

CONFIG_NAME += datetime.now().strftime("_%Y%m%d-%H%M%S")  # !!! give a meaningful name !!!
SAVE_DIR = RUNS_DIR / CONFIG_NAME

# Train

Start small (5e4 steps) to get fast feedback, then train from scratch on 5e5 - 1e6 steps
when you are confident in your setup. One step = one decision of the agent = 4 physics frames.

**TensorBoard**: in a terminal, run `uv run tensorboard --logdir runs` and open
http://localhost:6006. Look at:
- `landing/success_rate`: fraction of the last 100 training episodes that landed
- `landing/crash_rate`, `landing/abs_x_when_landed`, `landing/fuel_left_when_landed`
- `rollout/ep_rew_mean`: mean episode reward (only comparable between runs with the same reward!)

---
*RESTART FROM HERE WHEN CONTINUING TRAINING*

---

In [ ]:
TIMESTEPS = 5e5
LOAD = False  # True: continue training the last saved model instead of starting from scratch

if LOAD:
    model = type(model).load(SAVE_DIR / "last_model", env=env)

model.learn(
    total_timesteps=int(TIMESTEPS),
    tb_log_name=CONFIG_NAME,
    reset_num_timesteps=not LOAD,
    progress_bar=True,
    callback=CallbackList([
        LandingStatsCallback(),
        EvalCallback(eval_env, best_model_save_path=str(SAVE_DIR), eval_freq=25_000 // N_ENVS,
                     n_eval_episodes=20, deterministic=True, verbose=0),
    ]),
)
model.save(SAVE_DIR / "last_model")

# Evaluate

100 episodes with the deterministic policy (no exploration noise), on unseen starting
conditions. This is the score that counts.

In [ ]:
# best model found by the periodic evaluations during training, or the last one
MODEL_PATH = SAVE_DIR / "best_model" if (SAVE_DIR / "best_model.zip").exists() else SAVE_DIR / "last_model"
best = type(model).load(MODEL_PATH, device="cpu")

scores = evaluate(best, level=LEVEL, continuous=CONTINUOUS, n_episodes=100, config=CONFIG)
scores

## Visualize

The video is saved in `videos/`, keep it to track your progress.

In [ ]:
SEED = 0  # change it to see other starting conditions
path, traj = record_episode(best, VIDEOS_DIR / f"{CONFIG_NAME}_seed{SEED}.mp4", level=LEVEL,
                            continuous=CONTINUOUS, seed=SEED, config=CONFIG, reward_fn=REWARD_FN)
last = traj[-1]
print(f"steps: {len(traj)} | success: {last['landed_full']} | crash: {last['game_over']} "
      f"| x: {last['x']:+.3f} | fuel left: {last['fuel']:.0%}")
show(path)

## Observe what the agent does

Engine commands and state of the rocket along the episode above. Take a screenshot to
keep a record of your experiments.

In [ ]:
df = pd.DataFrame(traj)
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
df[["throttle", "gimbal", "side_thruster"]].plot(ax=axes[0], title="Commands")
df[["x", "y", "angle"]].plot(ax=axes[1], title="Position")
df[["velocity_x", "velocity_y", "fuel"]].plot(ax=axes[2], title="Speed and fuel")
axes[2].set_xlabel("time step")
plt.tight_layout()

In [ ]:
df[["reward", "base_reward"]].cumsum().plot(title="Cumulative reward (yours vs default)", figsize=(10, 3));